# 🧠 BrainScan AI — 01. Chargement et Prétraitement des Données IRM

---

### 📌 Contexte du Projet
**BrainScan AI** est une startup marocaine innovante spécialisée dans les technologies d'imagerie médicale assistée par l'intelligence artificielle. Notre mission est d'assister les radiologues et neurochirurgiens marocains dans le diagnostic précoce, précis et automatisé des tumeurs cérébrales à partir de coupes d'Imagerie par Résonance Magnétique (**IRM**).

### 🎯 Objectifs de ce Notebook
L'objectif de cette première phase est de construire un pipeline de données rigoureux, robuste et reproductible comprenant :
1. L'importation et la validation de l'environnement logiciel.
2. La configuration stricte des hyperparamètres d'ingestion (dimensions, extensions autorisées, répertoires).
3. L'inspection dynamique et le filtrage des classes anatomopathologiques.
4. Le chargement sécurisé des images IRM avec gestion d'erreurs (`try-except`), conversion colorimétrique RGB et redimensionnement standardisé.
5. Le mélange aléatoire (*shuffling*) pour éliminer tout biais d'ordre.
6. La transformation en tenseurs multidimensionnels NumPy (`float32`).
7. L'encodage catégoriel binarisé (*One-Hot Encoding*) des étiquettes.
8. La division stratifiée en sous-ensembles d'entraînement (**80%**) et de test (**20%**).
9. La normalisation des valeurs d'intensité de pixels dans l'intervalle $[0, 1]$.
10. La sérialisation sur disque au format binaire optimisé `.npy`.


## 1. Importation des Bibliothèques Nécessaires

### 💡 Ce que nous allons faire :
Nous importons l'ensemble des modules fondamentaux requis pour la manipulation de fichiers (`os`), la reproductibilité stochastique (`random`), la vision par ordinateur (`cv2`), le calcul matriciel (`numpy`), et la préparation des données sous `scikit-learn` et `tensorflow.keras`.

### ⚙️ Comment ça fonctionne et pourquoi :
* **`cv2` (OpenCV)** : Bibliothèque de vision industrielle de référence, utilisée pour lire et redimensionner efficacement les coupes IRM.
* **`numpy`** : Fournit les structures de tableaux à mémoire contiguë indispensables pour alimenter les réseaux de neurones profonds.
* **`train_test_split` & `LabelEncoder`** : Permettent respectivement la séparation stratifiée et la traduction des étiquettes textuelles en indices entiers.
* **`to_categorical`** : Convertit les entiers en vecteurs de probabilités One-Hot nécessaires pour la fonction de coût `categorical_crossentropy`.

### 🔍 Résultat attendu :
Toutes les dépendances se chargent sans avertissement bloquant (`warnings.filterwarnings("ignore")`). L'environnement de calcul est prêt.


In [1]:
import os
import random
import warnings

import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings("ignore")

2026-10-16 11:25:21.489969: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-16 11:25:21.521478: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-16 11:25:22.416598: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 2. Configuration des Paramètres et Chemins

### 💡 Ce que nous allons faire :
Nous définissons de manière centralisée les constantes globales du pipeline : le chemin d'accès aux images brutes, les dimensions cibles des images, les extensions de fichiers autorisées, la graine aléatoire de reproductibilité, et le dossier de destination pour les fichiers sérialisés.

### ⚙️ Comment ça fonctionne et pourquoi :
* **`IMG_SIZE = (224, 224)`** : Dimension standard en imagerie médicale et vision par ordinateur (compatible avec les architectures VGG/ResNet). Elle offre un excellent compromis entre résolution spatiale fine pour détecter de petites anomalies tissulaires et coût de calcul mémoire.
* **`VALID_EXTENSIONS`** : Liste blanche de sécurité (`.jpeg`, `.jpg`, `.bmp`, `.png`) rejetant d'éventuels fichiers corrompus, temporaires ou métadonnées parasites.
* **`RANDOM_SEED = 42`** : Fixe l'état du générateur pseudo-aléatoire pour garantir la stricte reproductibilité des résultats lors de chaque exécution.
* **`os.makedirs(OUTPUT_DIR, exist_ok=True)`** : Crée automatiquement l'arborescence `data/processed/` sans lever d'exception si elle existe déjà.

### 🔍 Résultat attendu :
Le répertoire `../data/processed` est créé ou vérifié, et toutes les variables de configuration sont instanciées en mémoire.


In [2]:
DATASET_DIR = "../data"
IMG_SIZE = (224, 224)
VALID_EXTENSIONS = {".jpeg", ".jpg", ".bmp", ".png"}
RANDOM_SEED = 42
TEST_SIZE = 0.2
OUTPUT_DIR = "../data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Exploration et Filtrage des Classes du Dataset

### 💡 Ce que nous allons faire :
Nous scannons dynamiquement le dossier `../data` pour extraire les répertoires correspondants aux classes tumorales, en excluant les dossiers techniques (comme `processed` ou les fichiers cachés `.`).

### ⚙️ Comment ça fonctionne et pourquoi :
L'arborescence du dataset classe les images médicales par dossier thématique. Le code vérifie `os.path.isdir` et filtre explicitement `processed` afin d'éviter qu'un dossier de cache ne soit interprété comme une cinquième classe tumorale.

### 🔍 Résultat attendu :
Affichage des 4 classes anatomopathologiques diagnostiques du projet :
1. `glioma` (Gliome)
2. `meningioma` (Méningiome)
3. `notumor` (Tissu cérébral sain / absence de tumeur)
4. `pituitary` (Adénome hypophysaire / pituitaire)


In [3]:
classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d)) and not d.startswith(".") and d != "processed"
])

print(f"Nombre de classes : {len(classes)}")
print(f"Classes : {classes}")

Nombre de classes : 4
Classes : ['glioma', 'meningioma', 'notumor', 'pituitary']


## 4. Chargement, Contrôle d'Intégrité et Prétraitement avec OpenCV

### 💡 Ce que nous allons faire :
Nous itérons sur chaque image de chaque classe pour valider son format, la charger depuis le disque, convertir son espace de couleur, la redimensionner aux dimensions cibles `(224, 224)`, et encapsuler chaque opération dans un bloc de capture d'exception `try-except`.

### ⚙️ Comment ça fonctionne et pourquoi :
* **Bloc `try-except`** : Protège le pipeline contre les fichiers corrompus ou tronqués qui feraient planter l'exécution globale. Si une image échoue, elle est comptabilisée dans `skipped` et le processus continue.
* **`cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`** : Par défaut, OpenCV charge les images dans l'ordre de canaux BGR (Bleu-Vert-Rouge). Les réseaux de neurones et les outils de visualisation (Matplotlib) s'attendent à du RGB. Cette conversion garantit la fidélité des contrastes tissulaires.
* **`cv2.resize(img, IMG_SIZE)`** : Aligne toutes les images hétérogènes du dataset sur la dimension uniforme 224 × 224 × 3.

### 🔍 Résultat attendu :
Chargement réussi de 7 023 images médicales valides, avec l'affichage du nombre exact d'images traitées et du nombre de fichiers ignorés (0 fichier corrompu).


In [4]:
images = []
labels = []
skipped = 0

for class_name in classes:
    class_dir = os.path.join(DATASET_DIR, class_name)
    files = os.listdir(class_dir)

    for fname in files:
        ext = os.path.splitext(fname)[1].lower()
        if ext not in VALID_EXTENSIONS:
            skipped += 1
            continue

        fpath = os.path.join(class_dir, fname)

        try:
            img = cv2.imread(fpath)
            if img is None:
                raise ValueError(f"OpenCV ne peut pas lire : {fpath}")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            images.append(img)
            labels.append(class_name)
        except Exception as e:
            print(f"Erreur lors du chargement de {fpath} : {e}")
            skipped += 1

print(f"Images chargées : {len(images)}")
print(f"Fichiers ignorés : {skipped}")

Images chargées : 7023
Fichiers ignorés : 0


## 5. Mélange Aléatoire des Données (Shuffling)

### 💡 Ce que nous allons faire :
Nous fusionnons temporairement les images et leurs étiquettes respectives sous forme de paires `(image, label)`, puis appliquons un mélange pseudo-aléatoire contrôlé par `RANDOM_SEED`.

### ⚙️ Comment ça fonctionne et pourquoi :
Lors du chargement séquentiel, les images sont ordonnées par classe (tous les gliomes, puis tous les méningiomes, etc.). Si nous n'effectuons pas de mélange, les mini-lots (*batches*) lors de l'entraînement ne contiendraient qu'une seule classe à la fois, provoquant de violentes oscillations du gradient et empêchant la convergence du réseau.

### 🔍 Résultat attendu :
Confirmation que la collection d'images et d'étiquettes conserve exactement la même taille (7 023 échantillons), désormais distribuée de manière homogène et aléatoire.


In [5]:
combined = list(zip(images, labels))
random.seed(RANDOM_SEED)
random.shuffle(combined)
images, labels = zip(*combined)

print(f"Total après mélange : {len(images)} images, {len(labels)} étiquettes")

Total après mélange : 7023 images, 7023 étiquettes


## 6. Conversion en Tenseurs Multidimensionnels NumPy

### 💡 Ce que nous allons faire :
Nous convertissons les listes Python d'images et d'étiquettes en tableaux matriciels NumPy optimisés (`np.ndarray`) avec le type de données `float32`.

### ⚙️ Comment ça fonctionne et pourquoi :
Les architectures Keras/TensorFlow opèrent sur des tenseurs contigus en mémoire. Le type `float32` (précision simple IEEE 754) est le standard absolu en Deep Learning : il accélère considérablement les calculs matriciels sur CPU et GPU tout en divisant l'occupation mémoire par deux par rapport au `float64`.

### 🔍 Résultat attendu :
Affichage des dimensions matricielles :
* $X$ : `(7023, 224, 224, 3)` — soit 7 023 images de 224 lignes, 224 colonnes et 3 canaux RGB.
* $y_{raw}$ : `(7023,)` — vecteur 1D contenant les noms de classes textuels.


In [6]:
X = np.array(images, dtype="float32")
y_raw = np.array(labels)

print(f"Shape de X : {X.shape}")
print(f"Shape de y_raw : {y_raw.shape}")

Shape de X : (7023, 224, 224, 3)
Shape de y_raw : (7023,)


## 7. Encodage Numérique et One-Hot des Étiquettes

### 💡 Ce que nous allons faire :
Nous convertissons les noms textuels de tumeurs (`glioma`, `meningioma`, etc.) en indices numériques entiers ($0, 1, 2, 3$), puis en vecteurs binaires *One-Hot*.

### ⚙️ Comment ça fonctionne et pourquoi :
* **`LabelEncoder`** : Assigne un index entier unique et ordonné alphabétiquement à chaque classe :
  * $0 \rightarrow$ `glioma`
  * $1 \rightarrow$ `meningioma`
  * $2 \rightarrow$ `notumor`
  * $3 \rightarrow$ `pituitary`
* **`to_categorical`** : Génère une matrice binaire où chaque ligne ne possède qu'un seul $1$ (à l'indice de la classe cible) et des $0$ ailleurs. C'est la forme mathématique requise par la couche de sortie `Dense(4, activation='softmax')`.

### 🔍 Résultat attendu :
Dictionnaire de correspondance clair des classes et dimension de `y_categorical` égale à `(7023, 4)`.


In [7]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
y_categorical = to_categorical(y_encoded, num_classes=len(le.classes_))

print(f"Classes encodées : {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Shape de y_categorical : {y_categorical.shape}")

Classes encodées : {np.str_('glioma'): np.int64(0), np.str_('meningioma'): np.int64(1), np.str_('notumor'): np.int64(2), np.str_('pituitary'): np.int64(3)}
Shape de y_categorical : (7023, 4)


## 8. Partitionnement Stratifié Train / Test (80% / 20%)

### 💡 Ce que nous allons faire :
Nous divisons l'ensemble des données en un jeu d'entraînement (**80%**, soit 5 618 images) et un jeu de test indépendant (**20%**, soit 1 405 images).

### ⚙️ Comment ça fonctionne et pourquoi :
* **`test_size = 0.2`** : Réserve 20% des données strictement pour l'évaluation finale non biaisée du modèle.
* **`stratify = y_encoded`** : **Essentiel en imagerie médicale**. La stratification garantit que la proportion de chaque classe tumorale est strictement identique dans le jeu d'entraînement et dans le jeu de test, évitant ainsi qu'une classe rare ne se retrouve sous-représentée dans le test.
* **`random_state = 42`** : Garantit une partition rigoureusement reproductible.

### 🔍 Résultat attendu :
Affichage des volumes d'échantillons : exactement 5 618 images d'entraînement et 1 405 images de test.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_encoded
)

print(f"Train : {X_train.shape[0]} images")
print(f"Test  : {X_test.shape[0]} images")

Train : 5618 images
Test  : 1405 images


## 9. Normalisation des Valeurs de Pixels dans $[0, 1]$

### 💡 Ce que nous allons faire :
Nous divisons toutes les valeurs d'intensité des pixels (initialement entières entre $0$ et $255$) par $255.0$ pour les ramener dans l'intervalle continu $[0.0, 1.0]$.

### ⚙️ Comment ça fonctionne et pourquoi :
La normalisation est cruciale pour l'optimisation par descente de gradient stochastique :
1. Elle empêche l'explosion ou la disparition des gradients dans les couches profondes.
2. Elle assure une surface de perte plus symétrique et sphérique, permettant à l'optimiseur Adam de converger beaucoup plus rapidement avec un taux d'apprentissage stable.

### 🔍 Résultat attendu :
Validation des bornes : valeur minimale $= 0.0000$ et valeur maximale $= 1.0000$.


In [9]:
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"Valeur min : {X_train.min():.4f} — Valeur max : {X_train.max():.4f}")

Valeur min : 0.0000 — Valeur max : 1.0000


## 10. Sauvegarde et Sérialisation des Tenseurs (.npy)

### 💡 Ce que nous allons faire :
Nous enregistrons les 4 tenseurs de données (`X_train`, `X_test`, `y_train`, `y_test`) ainsi que le vecteur des noms de classes (`label_classes.npy`) dans le dossier `../data/processed/`.

### ⚙️ Comment ça fonctionne et pourquoi :
Le format binaire natif NumPy (`.npy`) est extrêmement performant :
* Il préserve fidèlement les types (`float32`) et dimensions exactes sans perte de précision.
* Il permet un chargement quasi-instantané dans les notebooks suivants (EDA, entraînement, évaluation), évitant d'avoir à re-parcourir et re-traiter les 7 023 images individuelles à chaque session.

### 🔍 Résultat attendu :
Création des fichiers `.npy` dans `../data/processed/` prêts pour l'analyse exploratoire et la modélisation.


In [10]:
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)
np.save(os.path.join(OUTPUT_DIR, "label_classes.npy"), le.classes_)

print("Fichiers sauvegardés dans", OUTPUT_DIR)

Fichiers sauvegardés dans ../data/processed


## 11. 📋 Synthèse et Conclusion de la Phase 01

---

### ✅ Bilan des réalisations :
* **Intégrité du Dataset** : 7 023 coupes d'IRM cérébrales ont été inspectées, chargées et validées sans aucune perte ni corruption de fichier.
* **Prétraitement d'imagerie** : Conversion uniforme en RGB et redimensionnement strict en 224 × 224 pixels.
* **Encodage & Stratification** : 4 classes anatomiques parfaitement isolées (`glioma`, `meningioma`, `notumor`, `pituitary`) et réparties de manière équilibrée à 80% en entraînement (5 618 coupes) et 20% en test (1 405 coupes).
* **Normalisation & Persistance** : Échelle $[0, 1]$ appliquée avec succès et sauvegarde de l'ensemble des tenseurs dans `data/processed/`.

### ➡️ Prochaine étape :
Le notebook `02_eda_visualization.ipynb` prendra le relais pour explorer la distribution des classes, visualiser des échantillons d'IRM représentatifs, calculer le ratio de déséquilibre et dériver les poids de pénalisation (`class_weights`).
